In [ ]:
# =============================
# IMPORTS-1
# =============================
import pandas as pd
import yfinance as yf
import requests
from datetime import datetime, timedelta
import logging

logging.getLogger("yfinance").setLevel(logging.CRITICAL)

# =============================
# SETTINGS
# =============================
LOOKBACK_DAYS = 7

MIN_PRICE = 25
MAX_PRICE = 500

MIN_TREND = -0.05
MAX_TREND = 0.25   # 🚨 NEW: spike filter (reject overextended stocks)

VOL_MIN = 0.01
VOL_MAX = 0.12

MAX_DROP = -0.08

# =============================
# DATE LOGIC
# =============================
today = datetime.now().date()

def get_previous_market_day(date, n=1):
    count = 0
    while count < n:
        date -= timedelta(days=1)
        if date.weekday() < 5:
            count += 1
    return date

anchor_date = get_previous_market_day(today, 2)
start_date = anchor_date - timedelta(days=LOOKBACK_DAYS)

print(f"\n📅 Earnings window: {start_date} → {anchor_date}")

# =============================
# FETCH EARNINGS
# =============================
url = "https://stockcharts.com/json/data"
params = {"cmd": "get-reported-quarterly-earnings", "src": "freecharts-reported-earnings"}
headers = {"User-Agent": "Mozilla/5.0"}

data = requests.get(url, params=params, headers=headers).json()
earnings = data.get("earningsDates", [])

earnings_data = {}
earnings_names = {}

for e in earnings:
    raw_date = e.get("earningsDate")
    symbol = e.get("symbol")
    name = e.get("name")

    if not raw_date or not symbol:
        continue

    report_date = datetime.strptime(raw_date, "%Y-%m-%d").date()

    if start_date <= report_date <= anchor_date:
        earnings_data[symbol] = report_date
        earnings_names[symbol] = name

tickers = list(earnings_data.keys())
print(f"\n✅ Total confirmed earnings: {len(tickers)}")

# =============================
# STAGES
# =============================
weekly_stage, price_stage = [], []
trend_stage, reaction_stage, vol_stage = [], [], []
final_results = []

print("\n⚙️ Running filters...\n")

for symbol in tickers:
    try:
        stock = yf.Ticker(symbol)

        # =============================
        # WEEKLY OPTIONS
        # =============================
        expirations = stock.options

        has_weekly = any(
            (datetime.strptime(exp, "%Y-%m-%d").date() - today).days <= 10
            for exp in expirations[:5]
        )

        if not has_weekly:
            continue

        weekly_stage.append(symbol)

        # =============================
        # PRICE DATA
        # =============================
        hist = stock.history(period="3mo")
        if len(hist) < 50:
            continue

        price = hist["Close"].iloc[-1]

        # =============================
        # PRICE FILTER
        # =============================
        if not (MIN_PRICE <= price <= MAX_PRICE):
            continue

        price_stage.append(symbol)

        # =============================
        # TREND (WITH SPIKE FILTER)
        # =============================
        hist["SMA50"] = hist["Close"].rolling(50).mean()

        if pd.isna(hist["SMA50"].iloc[-1]):
            continue

        trend = (price - hist["SMA50"].iloc[-1]) / hist["SMA50"].iloc[-1]

        if trend < MIN_TREND or trend > MAX_TREND:
            continue

        trend_stage.append(symbol)

        # =============================
        # STABILITY
        # =============================
        recent = hist.tail(5)
        recent_return = (recent["Close"].iloc[-1] - recent["Close"].iloc[0]) / recent["Close"].iloc[0]

        if recent_return < MAX_DROP:
            continue

        reaction_stage.append(symbol)

        # =============================
        # VOLATILITY
        # =============================
        vol = hist["Close"].pct_change().std()

        if not (VOL_MIN <= vol <= VOL_MAX):
            continue

        vol_stage.append(symbol)

        # =============================
        # SCORING SYSTEM
        # =============================
        score = 0

        # Trend scoring (sweet spot)
        if 0.02 <= trend <= 0.10:
            score += 3
        elif -0.02 <= trend < 0.02:
            score += 2
        else:
            score += 1

        # Stability scoring
        if recent_return >= 0:
            score += 3
        elif recent_return > -0.03:
            score += 2
        else:
            score += 1

        # Volatility scoring (middle is best)
        if 0.02 <= vol <= 0.05:
            score += 3
        elif 0.01 <= vol < 0.02 or 0.05 < vol <= 0.08:
            score += 2
        else:
            score += 1

        # =============================
        # FINAL
        # =============================
        final_results.append({
            "Ticker": symbol,
            "Name": earnings_names[symbol],
            "Reported Date": earnings_data[symbol],
            "Price": round(price, 2),
            "Trend %": round(trend * 100, 2),
            "Volatility": round(vol, 4),
            "Score": score
        })

    except:
        continue

# =============================
# PRINT FUNCTION
# =============================
def print_stage(name, lst):
    print(f"\n{name} ({len(lst)} stocks):\n")
    if lst:
        print(pd.DataFrame({"Ticker": lst}))
        print("\n📋 TradingView List:\n")
        print(", ".join(lst))
    else:
        print("None")

# =============================
# OUTPUT STAGES
# =============================
print_stage("🟢 WEEKLY OPTIONS PASS", weekly_stage)
print_stage("💰 PRICE PASS", price_stage)
print_stage("📈 TREND PASS", trend_stage)
print_stage("🧠 STABILITY PASS", reaction_stage)
print_stage("⚡ VOLATILITY PASS", vol_stage)

# =============================
# FINAL OUTPUT + RANKING
# =============================
df_final = pd.DataFrame(final_results)

if df_final.empty:
    print("\n❌ No final setups")
else:
    df_sorted = df_final.sort_values(by="Score", ascending=False)

    print("\n🏆 RANKED CANDIDATES:\n")
    print(df_sorted)

    print("\n🔥 TOP 3 CANDIDATES:\n")
    print(df_sorted.head(3))

    print("\n📋 FINAL WATCHLIST:\n")
    print(", ".join(df_sorted["Ticker"]))

# =============================
# VERIFY LINK
# =============================
print("\n🔗 Verify Earnings Calendar:")
print("https://stockcharts.com/freecharts/reported-earnings.html")

# =============================
# STRATEGY GUIDE
# =============================
print("\n" + "="*60)
print("📘 OPTIONS STRATEGY EXECUTION GUIDE")
print("="*60)

print("""
ENTRY:
- Sell weekly put (~0.20 delta initially)
- Use defined risk: $2.5 or $5 wide spread (~8–10 weeks out)

SPREAD SELECTION:
- DEFAULT = $2.5 spread
- Upgrade to $5 ONLY if:
    ✔ Strong trend
    ✔ Clean earnings reaction
    ✔ Credit comfortably ≥ 10–15%

IMPORTANT:
- After Week 1:
  → DO NOT blindly follow 0.20 delta
  → Maintain risk structure
  → Only roll if credit justifies it

WEEKLY MANAGEMENT:

If price stays above strike:
→ Let expire
→ Sell next week

If price rises:
→ DO NOT aggressively move strike up
→ Only adjust if:
    ✔ Credit ≥ threshold
    ✔ Risk does NOT increase
    ✔ Strike remains ≤ 0.20 delta

If price drops near strike:
→ Roll ONLY if:
    ✔ Credit ≥ 10–15% of spread

CREDIT RULE (MANDATORY):
- Minimum credit:
    → ≥ 10% of spread width

- Preferred:
    → 12–16%

Examples:
- $2.5 spread → min $0.25
- $5 spread → min $0.50

❗ If below minimum:
→ SKIP trade

LONG PUT RULE:
- Do NOT auto-roll
- Only roll if:
    ✔ Total premium covers cost
    ✔ Position remains profitable

ASSIGNMENT:
- If assigned:
    → Sell long put (capture remaining value)
    → Close stock position

ONLY exercise if:
- No liquidity
- Near expiration (no time value)
- Emergency hedge needed

POSITION MANAGEMENT:
- Max 3 trades
- Add every ~3 weeks
- Avoid same sector overlap

CORE PRINCIPLE:
- Risk consistency > premium chasing
- Selectivity > frequency
- Skipping trades = edge
""")